# 第21章 数据读取与保存

掌握CSV与JSON的读取、参数控制、内存往返和结果导出。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 从文本读取表格
- 控制类型和缺失值
- 处理编码与日期
- 导出可复用结果


## 核心概念

- 读取时明确分隔符、编码、日期和类型。
- 保存索引前先判断它是否是业务字段。
- 浏览器课程使用内存文本模拟文件，方法与真实路径一致。


## 示例 1：读取CSV文本

StringIO让文本表现得像文件对象。


In [ ]:
from io import StringIO
import pandas as pd

csv_text = """order_id,region,amount,date
A1,华东,320.5,2026-01-05
A2,华南,880.0,2026-01-08
A3,华北,,2026-01-12
"""
orders = pd.read_csv(StringIO(csv_text), parse_dates=["date"])
print(orders)
print(orders.dtypes)


## 示例 2：控制缺失和类型

dtype和na_values使读取结果更稳定。


In [ ]:
csv_text = """product_id,category,stock
001,办公,18
002,数码,N/A
003,家居,24
"""
products = pd.read_csv(
    StringIO(csv_text),
    dtype={"product_id": "string", "category": "category"},
    na_values=["N/A"],
)
print(products)
print(products.dtypes)


## 示例 3：导出CSV与JSON

index=False避免把默认行号写成多余列。


In [ ]:
output_csv = orders.to_csv(index=False)
output_json = orders.to_json(orient="records", force_ascii=False, date_format="iso")
print(output_csv)
print(output_json)


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第21章 数据读取与保存”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 让商品编号被自动读取为整数并丢失前导零
- 保存默认索引造成Unnamed列
- 未检查日期和数值解析失败


## 综合练习

1. 读取一段分号分隔文本
2. 指定日期列
3. 筛选有效记录并导出CSV

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
from io import StringIO
import pandas as pd

text = """date;region;sales
2026-01-01;华东;120
2026-01-02;华南;98
2026-01-03;华北;invalid
"""

# TODO: 读取CSV文本，使用分号分隔符，解析日期列
data = pd.read_csv(StringIO(text), sep=";", parse_dates=["date"])

# TODO: 转换sales列为数值，无效值转为缺失值
data["sales"] = pd.to_numeric(data["sales"], errors="coerce")

# TODO: 删除sales列缺失的行
clean = data.dropna(subset=["sales"])

print(clean.to_csv(index=False))


In [ ]:
from io import StringIO
import pandas as pd

text = """date;region;sales
2026-01-01;华东;120
2026-01-02;华南;98
2026-01-03;华北;invalid
"""
data = pd.read_csv(StringIO(text), sep=";", parse_dates=["date"])
data["sales"] = pd.to_numeric(data["sales"], errors="coerce")
clean = data.dropna(subset=["sales"])
print(clean.to_csv(index=False))

# 自检
assert len(clean) == 2, "检查有效记录数：应该保留2条"
assert clean["sales"].sum() == 218, "检查有效销售额合计"


## 本章小结

掌握CSV与JSON的读取、参数控制、内存往返和结果导出。

**迁移思考**：

1. 如果 CSV 文件中的日期格式是"2026/01/05"而不是"2026-01-05"，pd.to_datetime 还能自动识别吗？
2. 为什么商品编号要用 dtype="string" 而不是让 Pandas 自动推断类型？


### 你已经掌握

- 从文本读取表格
- 控制类型和缺失值
- 处理编码与日期
- 导出可复用结果


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 读取CSV文本 | StringIO让文本表现得像文件对象。 | `pd.read_csv()` |
| 控制缺失和类型 | dtype和na_values使读取结果更稳定。 | `pd.read_csv()` |
| 导出CSV与JSON | index=False避免把默认行号写成多余列。 | `orders.to_csv()`、`orders.to_json()` |


### 需要注意

- 让商品编号被自动读取为整数并丢失前导零
- 保存默认索引造成Unnamed列
- 未检查日期和数值解析失败


### 完成检查

- [ ] 能够从文本读取表格
- [ ] 能够控制类型和缺失值
- [ ] 能够处理编码与日期
- [ ] 能够导出可复用结果
